# Laboratorio 5: Función de clasificación de tweets

Se utilizan el modelo y el vectorizador guardados en el notebook 04 (`best_model.pkl` y `tfidf_vectorizer.pkl`).

**Este notebook cubre:**
- Punto 7: Función que recibe un tweet sin preprocesar y devuelve si se refiere a un desastre real o no.

---

In [1]:
import warnings
warnings.filterwarnings("ignore")

import re
import string
import html
import joblib

import pandas as pd

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

pd.set_option("display.max_colwidth", 90)
RANDOM_STATE = 42

## 1. Carga del modelo y del vectorizador

In [2]:
modelo = joblib.load("./models/best_model.pkl")
vectorizador = joblib.load("./models/tfidf_vectorizer.pkl")

print("Modelo       :", modelo)
print("Vectorizador :", vectorizador)
print("Tamaño del vocabulario:", len(vectorizador.vocabulary_))

Modelo       : LogisticRegression(C=1, max_iter=1000, random_state=42)
Vectorizador : TfidfVectorizer(max_features=15000, min_df=2, ngram_range=(1, 2))
Tamaño del vocabulario: 9232


El mejor modelo del notebook 04 fue la Regresión Logística ajustada (`C = 1`, `solver = lbfgs`), entrenada sobre
una matriz TF-IDF de unigramas y bigramas. Al ser un modelo probabilístico expone `predict_proba`, lo que permite
devolver no solo la etiqueta sino también qué tan seguro está el modelo de su respuesta.

## 2. Reconstrucción del pipeline de limpieza

La función que recibe el tweet lo hará **sin preprocesar**, así que debe aplicarle exactamente la misma limpieza
que se usó para entrenar el modelo. Si el texto nuevo se limpiara de otra forma, los tokens no coincidirían con el
vocabulario del vectorizador y las predicciones perderían sentido. Por eso se replican tal cual las funciones
definidas en el notebook 01.

In [3]:
stop_words_en = set(stopwords.words("english"))


def corregir_mojibake(texto):
    texto = re.sub(r"\x89Û_", "'", texto)
    texto = re.sub(r"\x89Û[ÒÓªÏ]", " ", texto)
    texto = re.sub(r"åÊ|åÈ", " ", texto)
    texto = re.sub(r"[\x80-\x9f]", " ", texto)
    return texto


def quitar_urls(texto):
    return re.sub(r"https?://\S+|www\.\S+", " ", texto)


def quitar_menciones(texto):
    return re.sub(r"@\w+", " ", texto)


def procesar_hashtags(texto):
    return re.sub(r"#(\w+)", r"\1", texto)


def quitar_numeros_excepto_911(texto):
    tokens = texto.split()
    tokens = [t for t in tokens if not (t.isdigit() and t != "911")]
    return " ".join(tokens)


def limpiar_tweet(texto):
    texto = corregir_mojibake(texto)
    texto = quitar_urls(texto)
    texto = html.unescape(texto)
    texto = quitar_menciones(texto)
    texto = procesar_hashtags(texto)
    texto = texto.lower()
    texto = re.sub(f"[{re.escape(string.punctuation)}]", " ", texto)
    texto = quitar_numeros_excepto_911(texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    tokens = [w for w in texto.split() if w not in stop_words_en and len(w) > 1]
    return " ".join(tokens)

### 2.1 Verificación de que la limpieza es idéntica a la del notebook 01

In [4]:
df = pd.read_csv("./data/train_clean.csv")
df["clean_text"] = df["clean_text"].fillna("")

recalculado = df["text"].apply(limpiar_tweet)
coincidencia = (recalculado == df["clean_text"]).mean()

print(f"Tweets cuyo texto limpio coincide con el guardado en train_clean.csv: {coincidencia * 100:.2f}%")

Tweets cuyo texto limpio coincide con el guardado en train_clean.csv: 100.00%


La coincidencia es del 100%, lo que confirma que la función aplicada a texto crudo reproduce exactamente el
preprocesamiento con el que se entrenó el modelo.

## 3. Función de clasificación (Punto 7)

`clasificar_tweet` recibe el texto crudo de un tweet y ejecuta la cadena completa: limpieza, vectorización TF-IDF
y predicción. Devuelve un diccionario con el texto limpio intermedio (útil para depurar), la etiqueta, la
probabilidad de que sea desastre real y la confianza del modelo. El umbral es un parámetro, de modo que se puede
subir si interesa priorizar precisión o bajar si interesa priorizar recall.

In [5]:
ETIQUETAS = {0: "No desastre", 1: "Desastre real"}


def clasificar_tweet(tweet, umbral=0.5):
    """Recibe un tweet sin preprocesar y devuelve su clasificación."""
    tweet = "" if tweet is None else str(tweet)
    texto_limpio = limpiar_tweet(tweet)

    vector = vectorizador.transform([texto_limpio])
    prob_desastre = float(modelo.predict_proba(vector)[0, 1])
    prediccion = int(prob_desastre >= umbral)

    return {
        "tweet": tweet,
        "texto_limpio": texto_limpio,
        "prediccion": prediccion,
        "etiqueta": ETIQUETAS[prediccion],
        "prob_desastre": round(prob_desastre, 4),
        "confianza": round(max(prob_desastre, 1 - prob_desastre), 4),
    }


def clasificar_tweets(tweets, umbral=0.5):
    """Versión por lotes: devuelve un DataFrame con la clasificación de una lista de tweets."""
    return pd.DataFrame([clasificar_tweet(t, umbral) for t in tweets])

### 3.1 Ejemplo de uso sobre un tweet individual

In [6]:
resultado = clasificar_tweet("Breaking: 3 dead and 12 injured after a 6.2 earthquake hit the city #earthquake")

for clave, valor in resultado.items():
    print(f"{clave:14}: {valor}")

tweet         : Breaking: 3 dead and 12 injured after a 6.2 earthquake hit the city #earthquake
texto_limpio  : breaking dead injured earthquake hit city earthquake
prediccion    : 1
etiqueta      : Desastre real
prob_desastre : 0.8941
confianza     : 0.8941


### 3.2 Uso por lotes

In [7]:
ejemplos = [
    "Forest fire near La Ronge Sask. Canada",
    "Just happened a terrible car crash on the highway, ambulances everywhere",
    "Evacuation orders in place after the storm flooded the whole neighborhood",
    "Call 911 now, the building next door is burning",
    "I love this song so much, it is fire 🔥",
    "My exam went terrible today lol, total disaster",
    "@friend the new season of this show is a bomb, everyone is dying of laughter",
    "Watching a movie about a plane crash with my family tonight",
]

clasificar_tweets(ejemplos)[["tweet", "texto_limpio", "etiqueta", "prob_desastre"]]

,tweet,texto_limpio,etiqueta,prob_desastre
0,Forest fire near La Ronge Sask. Canada,forest fire near la ronge sask canada,Desastre real,0.9096
1,"Just happened a terrible car crash on the highway, ambulances everywhere",happened terrible car crash highway ambulances everywhere,Desastre real,0.6479
2,Evacuation orders in place after the storm flooded the whole neighborhood,evacuation orders place storm flooded whole neighborhood,Desastre real,0.7210
3,"Call 911 now, the building next door is burning",call 911 building next door burning,Desastre real,0.6016
4,"I love this song so much, it is fire 🔥",love song much fire,No desastre,0.1625
5,"My exam went terrible today lol, total disaster",exam went terrible today lol total disaster,No desastre,0.4584
6,"@friend the new season of this show is a bomb, everyone is dying of laughter",new season show bomb everyone dying laughter,No desastre,0.1812
7,Watching a movie about a plane crash with my family tonight,watching movie plane crash family tonight,Desastre real,0.7183


Los cuatro primeros ejemplos describen eventos reales con vocabulario factual de emergencia
(`fire`, `crash`, `evacuation`, `flooded`, `911`) y se clasifican correctamente como desastre. De los cuatro
últimos, el modelo acierta en los tres usos claramente figurados o coloquiales, aunque *"total disaster"* queda
cerca del umbral (0.46) porque esa palabra pesa mucho por sí sola. El último ejemplo se clasifica mal: *"plane
crash"* es un bigrama fuertemente asociado a desastre real y el modelo no capta que se habla de una película.
La columna `texto_limpio` muestra que menciones, URLs, hashtags y emojis se eliminan antes de vectorizar.

## 4. Casos difíciles y limitaciones

Vale la pena probar la función en casos donde el modelo tiene menos información o donde el lenguaje es ambiguo.

In [8]:
casos_dificiles = [
    "This traffic is killing me, I have been stuck here for two hours",
    "The crowd went wild, the stadium exploded when he scored",
    "@user http://t.co/abc123",
    "Aftershock",
    "hiroshima",
]

clasificar_tweets(casos_dificiles)[["tweet", "texto_limpio", "etiqueta", "prob_desastre", "confianza"]]

,tweet,texto_limpio,etiqueta,prob_desastre,confianza
0,"This traffic is killing me, I have been stuck here for two hours",traffic killing stuck two hours,Desastre real,0.6630,0.6630
1,"The crowd went wild, the stadium exploded when he scored",crowd went wild stadium exploded scored,No desastre,0.4476,0.5524
2,@user http://t.co/abc123,,No desastre,0.3847,0.6153
3,Aftershock,aftershock,No desastre,0.1688,0.8312
4,hiroshima,hiroshima,Desastre real,0.9536,0.9536


El modelo decide por vocabulario, así que las metáforas que reutilizan palabras de catástrofe son su
punto débil: *"traffic is killing me"* termina clasificado como desastre real con 0.66. El segundo caso sí se
resuelve bien, porque el resto de la frase (`crowd`, `stadium`, `scored`) empuja en dirección contraria. El tercero
es un tweet que queda vacío tras la limpieza, ya que solo contenía una mención y una URL: al convertirse en un
vector de ceros la predicción depende únicamente del intercepto del modelo (0.38) y termina en no desastre, que es
la decisión razonable cuando no hay evidencia. Los dos últimos muestran que una sola palabra puede bastar:
`hiroshima` es casi determinante hacia desastre, mientras que `aftershock` se clasifica como no desastre porque
todos los tweets del dataset con esa keyword hablan de un DJ y no de réplicas sísmicas. El modelo aprende el uso
de las palabras en este corpus, no su significado de diccionario.

## 5. Validación de la función sobre el conjunto de prueba

Para comprobar que la función funciona de extremo a extremo, se reproduce la división del notebook 04 con la misma
semilla y se clasifica el texto **crudo** del conjunto de prueba, es decir, sin usar la columna `clean_text` ya
calculada. Las métricas deben coincidir con las reportadas en el notebook 04.

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

_, texto_crudo_test, _, y_test = train_test_split(
    df["text"], df["target"], test_size=0.2, random_state=RANDOM_STATE, stratify=df["target"]
)

y_pred = texto_crudo_test.apply(lambda t: clasificar_tweet(t)["prediccion"])

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred, target_names=["No desastre", "Desastre real"]))

Accuracy: 0.8253
F1-score: 0.7798

               precision    recall  f1-score   support

  No desastre       0.81      0.90      0.86       869
Desastre real       0.85      0.72      0.78       654

     accuracy                           0.83      1523
    macro avg       0.83      0.81      0.82      1523
 weighted avg       0.83      0.83      0.82      1523



Las métricas son idénticas a las del notebook 04 (accuracy 0.8253 y F1 0.7798), lo que confirma que la función
replica el pipeline sin fugas ni diferencias de preprocesamiento.

### 5.1 Revisión de errores

In [10]:
comparacion = pd.DataFrame({
    "tweet": texto_crudo_test,
    "real": y_test.map(ETIQUETAS),
    "predicho": y_pred.map(ETIQUETAS),
})
errores = comparacion[comparacion["real"] != comparacion["predicho"]]

falsos_negativos = (errores["real"] == "Desastre real").sum()
falsos_positivos = (errores["real"] == "No desastre").sum()

print(f"Errores: {len(errores)} de {len(comparacion)} tweets de prueba "
      f"({len(errores) / len(comparacion) * 100:.1f}%)")
print(f"  Falsos negativos (desastre clasificado como no desastre): {falsos_negativos}")
print(f"  Falsos positivos (no desastre clasificado como desastre): {falsos_positivos}")
errores.sample(8, random_state=RANDOM_STATE)

Errores: 266 de 1523 tweets de prueba (17.5%)
  Falsos negativos (desastre clasificado como no desastre): 183
  Falsos positivos (no desastre clasificado como desastre): 83


,tweet,real,predicho
7242,PM Abe pledged to make every effort to seek a world without nuclear weapons. http://t....,Desastre real,No desastre
4154,You can never escape me. Bullets don't harm me. Nothing harms me. But I know pain. I k...,Desastre real,No desastre
7218,Agreed there - especially on automatic weapons. There's no legitimate reason for needi...,Desastre real,No desastre
895,Bloody insomnia again! Grrrr!! #Insomnia,Desastre real,No desastre
4742,@YoungHeroesID LAVA BLAST dan POWER RED #PantherAttack @Mirmanda11 @evaaaSR,Desastre real,No desastre
929,On #ThisDayInHistory in 1862 Confederate ship blown up by crew. Read More http://t.co/...,Desastre real,No desastre
6915,@canagal Good to hear it's back.. that storm's been given you guys trouble though :( ^SJ,Desastre real,No desastre
4973,Byproduct of metal price meltdown is a higher silver price http://t.co/cZWjw4UV7i,Desastre real,No desastre


Casi el 70% de los errores son falsos negativos, lo que es coherente con el recall de 0.72 de la
clase desastre real. En la muestra aparecen tweets que hablan de un desastre sin vocabulario explícito de
emergencia (declaraciones políticas, hechos históricos, titulares recortados por el límite de caracteres) y otros
cuyo contenido informativo estaba en la URL, que la limpieza elimina. Los falsos positivos son el caso inverso ya
visto en la sección 4: lenguaje figurado que reutiliza vocabulario de catástrofe. Es el mismo tipo de ambigüedad
detectado en el análisis exploratorio al revisar las palabras compartidas entre categorías.

## 6. Versión interactiva

Envoltorio para que el usuario escriba tweets por consola y reciba la clasificación en el momento. La llamada se
deja comentada para que el notebook pueda ejecutarse de principio a fin sin quedarse esperando entrada.

In [11]:
def clasificar_tweet_interactivo():
    """Pide tweets por consola y los clasifica hasta que el usuario escriba 'salir'."""
    print("Escriba un tweet para clasificarlo, o 'salir' para terminar.\n")
    while True:
        tweet = input("Tweet: ").strip()
        if tweet.lower() in {"salir", "exit", "quit", ""}:
            print("Fin.")
            break

        resultado = clasificar_tweet(tweet)
        print(f"  Texto limpio  : {resultado['texto_limpio']}")
        print(f"  Clasificación : {resultado['etiqueta']} "
              f"(probabilidad de desastre real: {resultado['prob_desastre']:.1%})\n")


# Descomentar para usarla:
# clasificar_tweet_interactivo()